In [ ]:
import json
import re
import fitz  # PyMuPDF
from typing import List, Dict, Optional, Tuple


# ============================================================
# 1. PDF EXTRACTION
# ============================================================
def extract_text_with_fitz(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in doc]
    return "\n".join(pages)


# ============================================================
# 2. CLEANING
# ============================================================
def clean_legal_pdf_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # --- Headers: covers BOTH formats ---
    # Format 1: "CÔNG BÁO/Số 977 + 978/Ngày 24-8-2024" (with +)
    # Format 2: "CÔNG BÁO/Số 175/Ngày 01-04-2026" (without +)
    # Also handles leading page number: "76 CÔNG BÁO/..."
    text = re.sub(
        r"\d{0,3}\s*CÔNG BÁO/Số\s+\d+(?:\s*\+\s*\d+)?/Ngày\s+\d{1,2}-\d{1,2}-\d{4}\s*",
        " ", text, flags=re.IGNORECASE
    )

    # --- Isolated page numbers ---
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)

    # --- Footnote blocks (for other laws that have them) ---
    text = re.sub(
        r"\n\s*\d{1,2}\s+(Luật An ninh mạng|Cụm từ|Điều\s+\d+\s+của\s+Luật).+?(?=\n(?:Điều\s+\d|Chương\s+[IVX]|\d+\.\s|[a-gđ]\))|$)",
        "\n", text, flags=re.DOTALL
    )

    # --- Inline superscript footnote markers ---
    text = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰]+", "", text)
    text = re.sub(r"(\w)(\d{1,2})(\s+(?:của|theo|giao|và|hoặc|trong))", r"\1\3", text)

    # --- Trailing continuation note ---
    text = re.sub(r"\(Xem tiếp Công báo[^\)]*\)", "", text)

    # --- Signature block ---
    text = re.sub(r"VĂN PHÒNG QUỐC HỘI.*$", "", text, flags=re.DOTALL)

    # --- Whitespace ---
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"^\s+", "", text, flags=re.MULTILINE)

    return text.strip()


# ============================================================
# 3. STRUCTURAL PARSING – with FALSE CHAPTER protection
# ============================================================

# Real chapter heading: "Chương X" followed by ALL-CAPS title on next line
CHAPTER_RE = re.compile(
    r"(Chương\s+[IVXLCDM]+)\s*\n\s*([A-ZĐÀ-Ỹ][A-ZĐÀ-Ỹ\s,]+)",
    flags=re.UNICODE
)

SECTION_RE = re.compile(
    r"(Mục\s+\d+)\s*\n\s*([A-ZĐÀ-Ỹ][^\n]+)",
    flags=re.UNICODE
)

ARTICLE_RE = re.compile(
    r"(Điều\s+\d+[a-z]?\.\s*[^\n]+)",
    flags=re.UNICODE
)

KHOAN_RE = re.compile(r"\n(?=\d{1,2}\.\s)")


def _find_spans(pattern, text) -> List[Tuple[int, int, str, str]]:
    results = []
    for m in pattern.finditer(text):
        results.append((m.start(), m.end(), m.group(1).strip(), m.group(2).strip()))
    return results


def parse_structure(text: str) -> List[Dict]:
    """
    Parse into articles with Chương/Mục metadata.
    Key fix: CHAPTER_RE requires ALL-CAPS title, preventing false matches
    like "Chương II\ncủa Luật này" inside article body text.
    """
    chapters = _find_spans(CHAPTER_RE, text)
    sections = _find_spans(SECTION_RE, text)
    article_matches = list(ARTICLE_RE.finditer(text))

    def _find_enclosing(pos: int, spans: list) -> Optional[str]:
        result = None
        for start, end, label, title in spans:
            if start <= pos:
                result = f"{label} {title}"
            else:
                break
        return result

    articles = []
    for i, m in enumerate(article_matches):
        art_start = m.start()
        art_end = article_matches[i + 1].start() if i + 1 < len(article_matches) else len(text)

        article_title = m.group(1).strip()
        article_body = text[m.end():art_end].strip()

        chapter_full = _find_enclosing(art_start, chapters)
        section_full = _find_enclosing(art_start, sections)

        articles.append({
            "article_title": article_title,
            "article_body": article_body,
            "chapter": chapter_full,
            "section": section_full,
        })

    return articles


# ============================================================
# 4. SMART SPLITTING & MERGING
# ============================================================

def split_article_by_khoan(title: str, body: str, max_len: int) -> List[str]:
    """
    Split long articles on khoản boundaries (1., 2., 3...).
    Every sub-chunk gets the article title prepended.
    """
    full_text = f"{title}\n{body}"

    if len(full_text) <= max_len:
        return [full_text]

    parts = KHOAN_RE.split(body)
    parts = [p.strip() for p in parts if p.strip()]

    chunks = []
    current = title

    for part in parts:
        candidate = f"{current}\n{part}" if current != title else f"{title}\n{part}"

        if len(candidate) <= max_len:
            current = candidate
        else:
            if current != title:
                chunks.append(current.strip())

            single = f"{title}\n{part}"
            if len(single) > max_len:
                # Hard split for extremely long single khoản
                offset = 0
                while offset < len(part):
                    end = min(offset + max_len - len(title) - 50, len(part))
                    if end < len(part):
                        for sep in ["\n", ". ", "; ", ", "]:
                            last_sep = part.rfind(sep, offset, end)
                            if last_sep > offset:
                                end = last_sep + len(sep)
                                break
                    chunk_text = part[offset:end].strip()
                    if chunk_text:
                        chunks.append(f"{title}\n{chunk_text}")
                    offset = end
                current = title
            else:
                current = single

    if current != title:
        chunks.append(current.strip())

    return chunks if chunks else [full_text[:max_len]]


def merge_short_articles(articles: List[Dict], min_len: int = 400) -> List[Dict]:
    """
    Merge consecutive short articles sharing the same Mục/Chapter.
    """
    if not articles:
        return articles

    merged = []
    buffer = None

    for art in articles:
        body_len = len(art["article_body"])

        if body_len >= min_len:
            if buffer:
                merged.append(buffer)
                buffer = None
            merged.append(art)
        else:
            if buffer is None:
                buffer = {**art}
            elif (buffer["chapter"] == art["chapter"]
                  and buffer["section"] == art["section"]):
                buffer["article_title"] += f" | {art['article_title']}"
                buffer["article_body"] += f"\n\n{art['article_title']}\n{art['article_body']}"
            else:
                merged.append(buffer)
                buffer = {**art}

    if buffer:
        merged.append(buffer)

    return merged


# ============================================================
# 5. MAIN PIPELINE
# ============================================================

def process_legal_pdf(
    pdf_path: str,
    source_file: str,
    law_name: Optional[str] = None,
    domain: Optional[str] = None,
    effective_date: Optional[str] = None,
    law_id: Optional[str] = None,
    max_chunk_len: int = 2800,
    merge_short: bool = True,
    min_merge_len: int = 400,
) -> List[Dict]:

    raw_text = extract_text_with_fitz(pdf_path)
    cleaned_text = clean_legal_pdf_text(raw_text)

    articles = parse_structure(cleaned_text)

    if merge_short:
        articles = merge_short_articles(articles, min_merge_len)

    final_chunks = []
    chunk_counter = 0

    for art in articles:
        title = art["article_title"]
        body = art["article_body"]
        chapter = art["chapter"]
        section = art.get("section")

        if not body.strip():
            continue

        hierarchy_parts = [p for p in [title, section, chapter] if p]
        combined_title = " | ".join(hierarchy_parts)

        sub_chunks = split_article_by_khoan(title, body, max_chunk_len)

        for idx, chunk_text in enumerate(sub_chunks, start=1):
            chunk_counter += 1
            chunk = {
                "chunk_id": f"{law_id or 'LAW'}_{chunk_counter:04d}",
                "law_name": law_name,
                "law_id": law_id,
                "source_file": source_file,
                "domain": domain,
                "effective_date": effective_date,
                "chapter": chapter,
                "section": section,
                "article": title,
                "title": combined_title,
                "context": chunk_text.strip(),
                "char_count": len(chunk_text.strip()),
            }
            if len(sub_chunks) > 1:
                chunk["sub_part"] = idx
                chunk["total_parts"] = len(sub_chunks)

            final_chunks.append(chunk)

    return final_chunks


# ============================================================
# 6. SAVE & DIAGNOSTICS
# ============================================================

def save_to_json(data: List[Dict], output_file: str):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def print_diagnostics(chunks: List[Dict]):
    lengths = [c["char_count"] for c in chunks]
    print(f"\n{'='*60}")
    print(f"  CHUNKING DIAGNOSTICS")
    print(f"{'='*60}")
    print(f"  Total chunks    : {len(chunks)}")
    print(f"  Avg length      : {sum(lengths)//len(lengths):,} chars")
    print(f"  Min length      : {min(lengths):,} chars")
    print(f"  Max length      : {max(lengths):,} chars")
    print(f"  Median length   : {sorted(lengths)[len(lengths)//2]:,} chars")

    brackets = [(0, 300), (300, 800), (800, 1500), (1500, 2800), (2800, 99999)]
    print(f"\n  Length distribution:")
    for lo, hi in brackets:
        count = sum(1 for l in lengths if lo <= l < hi)
        bar = "█" * count
        label = f"{lo}-{hi}" if hi < 99999 else f"{lo}+"
        print(f"    {label:>10s}: {count:3d} {bar}")

    multi = [c for c in chunks if "sub_part" in c]
    if multi:
        arts = set(c["article"] for c in multi)
        print(f"\n  Articles split into sub-parts ({len(arts)}):")
        for art in sorted(arts):
            parts = [c for c in multi if c["article"] == art]
            print(f"    {art[:60]:60s} → {len(parts)} parts")

    print(f"{'='*60}\n")


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    pdf_path = "E:\VN-legal-chatbot\data_processing\Luat_giao_thong.pdf"
    source_file = "Luat_giao_thong.pdf"
    law_name = "Luật Trật tự, an toàn giao thông đường bộ 2024"
    domain = "giao_thong"
    effective_date = "2025-01-01"
    law_id = "36_2024_QH15"
    output_json = "output_optimized_giao_thong.json"

    chunks = process_legal_pdf(
        pdf_path=pdf_path,
        source_file=source_file,
        law_name=law_name,
        domain=domain,
        effective_date=effective_date,
        law_id=law_id,
        max_chunk_len=2800,
    )

    print_diagnostics(chunks)

    save_to_json(chunks, output_json)
    print(f"Saved {len(chunks)} chunks to {output_json}")

<>:343: SyntaxWarning: invalid escape sequence '\V'
<>:343: SyntaxWarning: invalid escape sequence '\V'
C:\Users\Admin\AppData\Local\Temp\ipykernel_22384\1917267420.py:343: SyntaxWarning: invalid escape sequence '\V'
  pdf_path = "E:\VN-legal-chatbot\data_processing\Luat_giao_thong.pdf"



  CHUNKING DIAGNOSTICS
  Total chunks    : 29
  Avg length      : 1,334 chars
  Min length      : 253 chars
  Max length      : 2,793 chars
  Median length   : 1,234 chars

  Length distribution:
         0-300:   1 █
       300-800:  11 ███████████
      800-1500:   7 ███████
     1500-2800:  10 ██████████
         2800+:   0 

  Articles split into sub-parts (5):
    Điều 11. Chấp hành báo hiệu đường bộ                         → 2 parts
    Điều 18. Dừng xe, đỗ xe                                      → 2 parts
    Điều 2. Giải thích từ ngữ                                    → 2 parts
    Điều 4. Chính sách của Nhà nước về trật tự, an toàn giao thô → 2 parts
    Điều 9. Các hành vi bị nghiêm cấm                            → 3 parts

Saved 29 chunks to output_optimized_giao_thong.json
